In [2]:
# Load packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

Load Data

In [4]:
import pandas as pd

# Load datasets
old_data = pd.read_csv('combined_data.csv')
new_data = pd.read_csv('new_car_data.csv')

# Normalize column names
old_data.columns = old_data.columns.str.strip().str.lower().str.replace(' ', '_')
new_data.columns = new_data.columns.str.strip().str.lower().str.replace(' ', '_')

# Convert timestamps
old_data['activity_start_timestamp'] = pd.to_datetime(
    old_data['activity_start_timestamp'], format='mixed', errors='coerce'
)
new_data['activity_start_timestamp'] = pd.to_datetime(
    new_data['activity_start_timestamp'], format='mixed', errors='coerce'
)

/tmp/ipython-input-1472465322.py:4: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  old_data = pd.read_csv('combined_data.csv')


In [12]:
# Convert new data to datetime
new_data['activity_start_timestamp'] = pd.to_datetime(new_data['activity_start_timestamp'], errors='coerce')

# Remove timezone info
new_data['activity_start_timestamp'] = new_data['activity_start_timestamp'].dt.tz_localize(None)

# Check
print(new_data['activity_start_timestamp'].head())

0   2025-10-01 00:25:00
1   2025-10-01 00:25:00
2   2025-10-01 00:25:00
3   2025-10-01 00:25:00
4   2025-10-01 00:25:00
Name: activity_start_timestamp, dtype: datetime64[ns]


In [14]:
# Oct 1-14 did not include any seconds, so we exclude those 2 weeks from the analysis so our results are not skewed
# Define the cutoff date
cutoff_date = pd.to_datetime('2025-10-15 00:00:00')

# Filter new_data
new_data = new_data[new_data['activity_start_timestamp'] >= cutoff_date]

# Reset the index
new_data = new_data.reset_index(drop=True)

# Check
print(new_data['activity_start_timestamp'].min(), new_data['activity_start_timestamp'].max())

2025-10-15 04:11:18 2025-11-30 21:00:17


# Pre vs. Post Implementation Analysis
Goal: For each call, compute the time from:
- SeniorsMenu -> SuburbsOrCityMenu

Before the implementation, callers had to go thorugh an addition menu: SeniorsConfirmation. The assumption is that the time to get from SeniorsMenu to SuburbsOrCityMenu should be greatly reduced following the implementation on October 16, 2025 of taking the SeniorsConfirmation menu out.

In [16]:
# Define implementation date
cutoff = pd.to_datetime('2025-10-16 00:00:00')

# Split pre and post implementation
pre_data = old_data[old_data['activity_start_timestamp'] < cutoff]

post_data = new_data[new_data['activity_start_timestamp'] >= cutoff]

# Sort both datasets
pre_data = pre_data.sort_values(
    ['contact_session_id', 'activity_start_timestamp']
).reset_index(drop=True)

post_data = post_data.sort_values(
    ['contact_session_id', 'activity_start_timestamp']
).reset_index(drop=True)

In [17]:
def senior_menu_time(df):
    # Filter for rows containing the menu items
    seniors = df[df['activity_name'] == 'SeniorsMenu'][['contact_session_id', 'activity_start_timestamp']]
    suburbs = df[df['activity_name'] == 'SuburbsOrCityMenu'][['contact_session_id', 'activity_start_timestamp']]

    # Get the first occurrence for each Contact Session ID
    seniors_first = seniors.groupby('contact_session_id')['activity_start_timestamp'] \
        .min().rename('seniorsmenu_time')

    suburbs_first = suburbs.groupby('contact_session_id')['activity_start_timestamp'] \
        .min().rename('suburbsorcitymenu_first_time')

    suburbs_last = suburbs.groupby('contact_session_id')['activity_start_timestamp'] \
        .max().rename('suburbsorcitymenu_last_time')

    # Merge timestamps
    times = pd.merge(seniors_first, suburbs_first, left_index=True, right_index=True, how='inner')
    time2 = pd.merge(seniors_first, suburbs_last, left_index=True, right_index=True, how='inner')

    # Calculate time differences
    times['time_diff_first'] = (
        times['suburbsorcitymenu_first_time'] - times['seniorsmenu_time']
    ).dt.total_seconds()

    time2['time_diff_last'] = (
        time2['suburbsorcitymenu_last_time'] - time2['seniorsmenu_time']
    ).dt.total_seconds()

    # Merge the diff timeframes
    combined = pd.merge(times, time2[['time_diff_last']],
                        left_index=True, right_index=True, how='outer')

    # First timestamp for each session
    first_activity = df.groupby('contact_session_id')['activity_start_timestamp'].min()

    # Merge into final dataframe
    dataframe = pd.merge(
        combined,
        first_activity.rename('activity_start_timestamp'),
        left_index=True,
        right_index=True,
        how='left'
    ).reset_index()

    return dataframe


# Apply to the old + new dataframes
old_times = senior_menu_time(old_data)
new_times = senior_menu_time(new_data)

In [18]:
new_times.head()

,contact_session_id,seniorsmenu_time,suburbsorcitymenu_first_time,time_diff_first,time_diff_last,activity_start_timestamp
0,003ce994-b111-45e9-9c62-279f592c0ab4,2025-10-16 09:14:32,2025-10-16 09:14:41,9.0,130.0,2025-10-16 09:06:51
1,0074ed90-042a-4e2a-b913-13163d1b8fa9,2025-11-13 08:03:51,2025-11-13 08:03:58,7.0,7.0,2025-11-13 08:00:18
2,008ffe6f-b0af-49ac-9c38-0d567db76870,2025-11-05 10:21:04,2025-11-05 10:21:10,6.0,6.0,2025-11-05 10:18:00
3,009179f7-6646-4b78-9965-423bb5af4f5e,2025-10-15 11:33:36,2025-10-15 11:33:42,6.0,6.0,2025-10-15 11:31:44
4,00e68d1b-dad9-44f0-81c7-74709f818d40,2025-11-18 10:58:27,2025-11-18 10:58:35,8.0,8.0,2025-11-18 10:55:26


In [23]:
# Pre-implementation averages
pre_avg_first = np.mean(old_times['time_diff_first'])

# Post-implementation averages
post_avg_first = np.mean(new_times['time_diff_first'])

print(f"Pre-implementation: {pre_avg_first:.2f} seconds")
print(f"Post-implementation: {post_avg_first:.2f} seconds")

Pre-implementation: 23.74 seconds
Post-implementation: 7.16 seconds
